In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd
import os

from concurrent.futures import ThreadPoolExecutor, as_completed


# 대기시간
st = 3


# 검색 키워드
keywords = [
    "에어컨",
    "선풍기",
    "제습기",
    "냉풍기",
    "서큘레이터"
]



# ==========================================
# 하나의 키워드 검색 함수
# ==========================================

def naver_cafe_search(keyword):

    result = []


    print("="*60)
    print(f"{keyword} 검색 시작")
    print("="*60)



    # 크롬 실행
    driver = webdriver.Chrome()


    try:

        # 네이버 접속
        driver.get("https://www.naver.com")
        time.sleep(st)



        # 검색창
        green_box_ele = driver.find_element(
            By.ID,
            "query"
        )


        # 검색어 입력
        green_box_ele.send_keys(keyword)



        # 검색 버튼 클릭
        ai_s_else = driver.find_element(
            By.CLASS_NAME,
            "ai_effect_symbol"
        )

        ai_s_else.click()

        time.sleep(st)



        # 카페 메뉴 클릭
        elements = driver.find_elements(
            By.CLASS_NAME,
            "wxxES_QvNoYHFNCZ"
        )


        for element in elements:

            if element.text.strip() == "카페":

                element.click()

                print(
                    f"{keyword} 카페 클릭 완료"
                )

                break


        time.sleep(st)



        # 게시글 가져오기
        title_links = driver.find_elements(
            By.CLASS_NAME,
            "title_link"
        )



        for element in title_links:


            title = element.text.strip()

            url = element.get_attribute(
                "href"
            )


            print(
                keyword,
                title
            )


            result.append(
                {
                    "키워드": keyword,
                    "게시글제목": title,
                    "URL": url
                }
            )


    except Exception as e:

        print(
            keyword,
            "오류 발생 :",
            e
        )


    finally:

        # 브라우저 종료
        driver.quit()



    print(
        f"{keyword} 검색 완료"
    )


    return result





# ==========================================
# 병렬 실행
# ==========================================


total_result = []



# 동시에 실행할 브라우저 개수
with ThreadPoolExecutor(max_workers=5) as executor:


    futures = []


    for keyword in keywords:

        futures.append(
            executor.submit(
                naver_cafe_search,
                keyword
            )
        )



    # 결과 수집
    for future in as_completed(futures):

        data = future.result()

        total_result.extend(data)




# ==========================================
# Excel 저장
# ==========================================


df = pd.DataFrame(
    total_result
)



save_path = r"D:\Edu\Machine Learning\workspace\20.업무자동화\결과\웹"



if not os.path.exists(save_path):

    os.makedirs(save_path)



file_path = os.path.join(
    save_path,
    "naver_cafe_elect02.xlsx"
)



df.to_excel(
    file_path,
    index=False
)



print("="*70)
print("Excel 저장 완료")
print(file_path)
print("="*70)

print(df)